# 03 生成已走完的周 K

每个已走完的周一张图，不是每个交易日一张。周二用的就是上周那张，周五用的是本周这张。日 K 不再生成，直接读 `k_pictures_gray_body`。

图写到 `k_pictures_gray_fuse_week`。不要写入 `k_pictures_gray_body`、`k_pictures_gray_week`、`k_pictures_rgb_I20`。

先 `MAX_CODES = 20`。全量再改 `None`。

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "change_rgb_i20"]
candidates.extend(cwd.parents)
for parent in [cwd, *cwd.parents]:
    candidates.append(parent / "change_rgb_i20")
seen = set()
root = None
for p in candidates:
    key = str(p)
    if key in seen:
        continue
    seen.add(key)
    if (p / "helpers.py").exists() and (p / "draw_gray.py").exists() and (p / "images_lib.py").exists():
        root = p
        break
if root is None:
    raise FileNotFoundError(
        f"找不到 helpers.py、draw_gray.py、images_lib.py。cwd={cwd}。"
        "这一格只定位代码目录，窗口在下一格改 WINDOW_KEY。"
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from helpers import notebook_bootstrap
from images_lib import generate_images, verify_images

ROOT, cfg, paths = notebook_bootstrap()
print("ROOT      =", ROOT)
print("processed =", paths["processed"])
print("images    =", paths["images"])
print("remote    =", paths.get("images_remote"))

## 参数

- `WINDOW_KEY`：`20`（100×60）或 `60`（100×180）
- `MAX_CODES`：调试用前 N 只；全量改 `None`（全量自动写 144 服务器）
- `USE_REMOTE`：`"auto"` 全量走 144；调试走本地 `data/images`
- `COUNT_ONLY=True` 只统计样本数、不写 `images.npy`
- `JOBS=20`：进程池用 20 核

In [ ]:
import importlib
import inspect
import images_lib

importlib.reload(images_lib)
generate_images = images_lib.generate_images
verify_images = images_lib.verify_images
if "window_key" not in inspect.signature(generate_images).parameters:
    raise RuntimeError(
        f"加载到的是旧 images_lib：{images_lib.__file__}\n"
        "generate_images 还只有 cfg 和 paths 两个位置参数。\n"
        "用这一版 change_rgb_i20/images_lib.py 覆盖服务器上的文件，然后再跑这一格。"
    )

WINDOW_KEY = 20            # 20 或 60；也可以写 "I20" / "I60"
MAX_CODES = 20
DEBUG_CODES = None          # 例如 ["000001", "600519"]
COUNT_ONLY = False
JOBS = 20
USE_REMOTE = "auto"

info = generate_images(
    cfg, paths, window_key=WINDOW_KEY,
    max_codes=MAX_CODES,
    debug_codes=DEBUG_CODES,
    count_only=COUNT_ONLY,
    jobs=JOBS,
    require_volume=True,
    use_remote=USE_REMOTE,
)
print(info)

## 校验 + 25 张样例拼图

拼图在 `figures/verify_{WINDOW_KEY}.png`。日期应该都是周末最后一个交易日。20 只通过后再把 `MAX_CODES` 改成 `None`。

In [ ]:
if not COUNT_ONLY:
    verify_images(cfg, paths, WINDOW_KEY, img_dir=info.get("img_dir"))
else:
    print("count-only，跳过 verify")